In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [4]:
# ============================================================
# PHASE 1 - CELL 1: Kiểm tra môi trường và trạng thái GPU
# Mục đích:
# - Đảm bảo Notebook đang chạy trên GPU T4 của Kaggle.
# - Kiểm tra phiên bản PyTorch và VRAM (dung lượng bộ nhớ GPU).
# - Log ra thông tin phần cứng để dễ debug nếu gặp lỗi OOM (Out of Memory) sau này.
# ============================================================

import sys
import platform

def check_environment():
    print("=" * 60)
    print(f"Python      : {sys.version.split()[0]}")
    print(f"Platform    : {platform.platform()}")
    
    # Kiểm tra thư viện PyTorch (Kaggle luôn có sẵn)
    try:
        import torch
        print(f"PyTorch     : {torch.__version__}")
        
        # Kiểm tra GPU
        cuda_available = torch.cuda.is_available()
        print(f"CUDA avail  : {cuda_available}")
        
        if cuda_available:
            print(f"GPU Name    : {torch.cuda.get_device_name(0)}")
            props = torch.cuda.get_device_properties(0)
            # VRAM tính bằng GB, làm tròn 2 chữ số thập phân
            print(f"VRAM        : {props.total_memory / 1024**3:.2f} GB")
            print(f"Compute Cap : {props.major}.{props.minor}")
        else:
            print("⚠️ CẢNH BÁO: Không tìm thấy GPU. Hãy vào 'Settings' -> 'Accelerator' và chọn 'GPU T4 x2'")
            
    except ImportError:
        print("[!] PyTorch chưa được cài đặt.")
        
    print("=" * 60)

check_environment()

Python      : 3.13.16
Platform    : Linux-6.18.55+-x86_64-with-glibc2.39
PyTorch     : 2.11.0+cu130
CUDA avail  : True
GPU Name    : Tesla T4
VRAM        : 14.56 GB
Compute Cap : 7.5


In [5]:
# ============================================================
# PHASE 1 - CELL 2: Tự động kiểm tra & cài đặt thư viện
# Mục đích:
# - Kaggle đã cài sẵn transformers, datasets, accelerate. 
# - Tuy nhiên, thư viện 'seqeval' (dùng để tính F1-score NER) thường bị thiếu.
# - Hàm này sẽ kiểm tra, nếu thiếu mới pip install để tiết kiệm thời gian.
# ============================================================

import importlib
import subprocess
import sys

# Map: tên package trên PyPI -> tên module dùng để import
REQUIRED_PACKAGES = {
    "seqeval"     : "seqeval",       # Đánh giá NER bằng Precision/Recall/F1
    "transformers": "transformers",  # Thư viện Hugging Face (PhoBERT)
    "datasets"    : "datasets",      # Xử lý dataset
    "accelerate"  : "accelerate",    # Hỗ trợ huấn luyện (mixed precision, v.v.)
    "scikit-learn": "sklearn",       # Chia tập dữ liệu, tính toán metric
}

def ensure_packages(package_map: dict):
    """Kiểm tra và cài đặt các package còn thiếu."""
    missing = []
    print("Đang kiểm tra thư viện...")
    
    for pip_name, import_name in package_map.items():
        try:
            importlib.import_module(import_name)
            print(f"[✓] {pip_name:15s} đã có sẵn")
        except ImportError:
            print(f"[x] {pip_name:15s} THIẾU → sẽ cài đặt")
            missing.append(pip_name)

    # Chỉ chạy pip install nếu thực sự có gói bị thiếu
    if missing:
        print(f"\n>>> Đang cài đặt: {missing}")
        # --no-cache-dir giúp tiết kiệm ổ đĩa trên Kaggle
        cmd = [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir"] + missing
        subprocess.check_call(cmd)
        print(">>> Cài đặt hoàn tất!\n")
    else:
        print("\n>>> Tất cả thư viện đã sẵn sàng, không cần cài thêm.\n")

# Thực thi
ensure_packages(REQUIRED_PACKAGES)

Đang kiểm tra thư viện...
[✓] seqeval         đã có sẵn
[✓] transformers    đã có sẵn
[✓] datasets        đã có sẵn
[✓] accelerate      đã có sẵn
[✓] scikit-learn    đã có sẵn

>>> Tất cả thư viện đã sẵn sàng, không cần cài thêm.



In [8]:
# ============================================================
# PHASE 1 - CELL 3: Khai báo cấu hình (Config) & Đường dẫn
# Mục đích:
# - Gom toàn bộ đường dẫn, nhãn, và tham số vào một Class duy nhất.
# - Chuyển bài toán từ 3 nhãn BIO (O, B-SKILL, I-SKILL) sang 2 nhãn (O, SKILL).
# ============================================================

from pathlib import Path

class Config:
    # ---- ĐƯỜNG DẪN ----
    # Đường dẫn chính xác đến file dataset trên Kaggle
    DATA_PATH  = Path("/kaggle/input/datasets/huyende/vn-resumes-ner-txt/Vn_resumes_NER.txt")
    # Thư mục lưu các file kết quả (mô hình, splits, log...)
    OUTPUT_DIR = Path("/kaggle/working")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # ---- NHÃN (BINARY CLASSIFICATION) ----
    # Dataset gốc dùng BIO. Ta map B/I-SKILL thành SKILL để đánh giá theo 2 nhãn.
    LABEL_MAP = {
        "O": "O", 
        "B-SKILL": "SKILL", 
        "I-SKILL": "SKILL"
    }
    LABELS    = ["O", "SKILL"]                        # 2 nhãn đích
    LABEL2ID  = {lbl: i for i, lbl in enumerate(LABELS)}  # O=0, SKILL=1
    ID2LABEL  = {i: lbl for lbl, i in LABEL2ID.items()}

    # ---- CHIA TẬP (TRAIN / VAL / TEST) ----
    TRAIN_RATIO = 0.80
    VAL_RATIO   = 0.10
    TEST_RATIO  = 0.10
    SEED        = 42

    # ---- THAM SỐ MÔ HÌNH ----
    MODEL_NAME  = "vinai/phobert-base" # Sẽ dùng ở Phase 2 & 3
    MAX_LEN     = 256                  # Độ dài tối đa của chuỗi (PhoBERT max=256)

# ---- In ra để kiểm tra ----
print("=" * 50)
print("KIỂM TRA CẤU HÌNH")
print("=" * 50)
print(f"Data path    : {Config.DATA_PATH}")
print(f"File tồn tại : {Config.DATA_PATH.exists()}")
print(f"Nhãn dự án   : {Config.LABELS}")
print(f"Ánh xạ nhãn  : {Config.LABEL2ID}")
print(f"Model dự kiến: {Config.MODEL_NAME}")
print(f"Max length   : {Config.MAX_LEN}")
print("=" * 50)

KIỂM TRA CẤU HÌNH
Data path    : /kaggle/input/datasets/huyende/vn-resumes-ner-txt/Vn_resumes_NER.txt
File tồn tại : True
Nhãn dự án   : ['O', 'SKILL']
Ánh xạ nhãn  : {'O': 0, 'SKILL': 1}
Model dự kiến: vinai/phobert-base
Max length   : 256


In [9]:
# ============================================================
# PHASE 1 - CELL 4: Hàm đọc file NER & Chuẩn hóa
# Mục đích:
# - Đọc file Vn_resumes_NER.txt có định dạng <s> token nhãn </s>
# - Chuẩn hóa Unicode về NFC để tránh lỗi token bị tách khi dùng PhoBERT.
# - Chuyển nhãn gốc (O, B-SKILL, I-SKILL) sang nhãn đích (O, SKILL).
# ============================================================

import unicodedata
from typing import List, Tuple

def normalize_unicode(text: str, form: str = "NFC") -> str:
    """
    Chuẩn hóa chuỗi Unicode về dạng NFC (Canonical Composition).
    - Ví dụ: 'ệ' ở NFC là 1 codepoint, ở NFD là 'e' + dấu mũ + dấu nặng.
    - Giúp đồng nhất dữ liệu đầu vào cho Tokenizer.
    """
    if not text:
        return text
    return unicodedata.normalize(form, text)


def read_ner_file(file_path: Path) -> List[Tuple[List[str], List[str]]]:
    """
    Đọc file NER định dạng CoNLL có thẻ <s>...</s>.
    Trả về: List các câu, mỗi câu là Tuple (danh sách tokens, danh sách nhãn).
    """
    if not file_path.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {file_path}")

    sentences = []
    cur_tokens, cur_tags = [], []
    inside_sentence = False
    
    skipped_lines = 0
    unknown_tags = 0

    with open(file_path, "r", encoding="utf-8") as f:
        for line_no, raw_line in enumerate(f, start=1):
            line = raw_line.rstrip("\n")

            # 1. Xử lý thẻ mở/đóng câu
            if line.strip().lower() == "<s>":
                inside_sentence = True
                cur_tokens, cur_tags = [], []
                continue
                
            if line.strip().lower() == "</s>":
                inside_sentence = False
                if cur_tokens:
                    sentences.append((cur_tokens, cur_tags))
                cur_tokens, cur_tags = [], []
                continue

            # 2. Bỏ qua dòng trống
            if not line.strip():
                continue

            # 3. Tách token và nhãn (ưu tiên tab, nếu không có dùng space)
            parts = line.split("\t")
            if len(parts) != 2:
                parts = line.rsplit(maxsplit=1)
            if len(parts) != 2:
                skipped_lines += 1
                continue

            token, raw_tag = parts[0].strip(), parts[1].strip()

            if not inside_sentence:
                continue

            # 4. Map nhãn BIO -> nhãn đích (O, SKILL)
            if raw_tag not in Config.LABEL_MAP:
                unknown_tags += 1
                continue
            tag = Config.LABEL_MAP[raw_tag]

            # 5. Chuẩn hóa token NFC và lưu
            cur_tokens.append(normalize_unicode(token))
            cur_tags.append(tag)

    # Xử lý trường hợp file thiếu thẻ </s> ở cuối
    if cur_tokens:
        sentences.append((cur_tokens, cur_tags))

    print(f"[✓] Đọc thành công: {len(sentences):,} câu.")
    print(f"    Dòng không hợp lệ: {skipped_lines}")
    print(f"    Nhãn lạ bị bỏ qua: {unknown_tags}")
    
    return sentences


# ---- Chạy thử ----
print("Đang parse dữ liệu...")
data = read_ner_file(Config.DATA_PATH)

# In thử 1 câu mẫu để kiểm tra
if data:
    tokens, tags = data[0]
    print("\n--- Câu mẫu #1 ---")
    print(f"Tokens: {tokens[:10]}..." if len(tokens) > 10 else f"Tokens: {tokens}")
    print(f"Tags  : {tags[:10]}..." if len(tags) > 10 else f"Tags  : {tags}")
    print(f"Độ dài Tokens = Độ dài Tags: {len(tokens) == len(tags)}")

Đang parse dữ liệu...
[✓] Đọc thành công: 19,323 câu.
    Dòng không hợp lệ: 0
    Nhãn lạ bị bỏ qua: 0

--- Câu mẫu #1 ---
Tokens: ['tạo', 'các', 'module', 'agent', ',', 'callcenter', ',', 'customer', 'trên', 'vtiger']...
Tags  : ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'SKILL']...
Độ dài Tokens = Độ dài Tags: True


In [10]:
# ============================================================
# PHASE 1 - CELL 5: Thống kê phân bố nhãn
# Mục đích:
# - Đếm số lượng token của từng nhãn (O và SKILL).
# - Tính tỷ lệ phần trăm để đánh giá mức độ mất cân bằng (imbalance).
# - Nếu SKILL chiếm < 10%, model rất dễ bị "học vẹt" (đoán toàn bộ là O).
# ============================================================

from collections import Counter

def label_statistics(data):
    """Hàm đếm và in báo cáo phân bố nhãn."""
    counter = Counter()
    total_tokens = 0
    
    # Duyệt qua toàn bộ dữ liệu để đếm
    for tokens, tags in data:
        counter.update(tags)
        total_tokens += len(tokens)

    print("=" * 55)
    print(f"{'NHÃN':<12}{'SỐ LƯỢNG':>14}{'TỶ LỆ':>12}")
    print("-" * 55)
    
    for lbl in Config.LABELS:
        cnt = counter.get(lbl, 0)
        pct = 100.0 * cnt / max(total_tokens, 1)
        print(f"{lbl:<12}{cnt:>14,}{pct:>11.2f}%")
        
    print("-" * 55)
    print(f"{'TỔNG':<12}{total_tokens:>14,}{'100.00%':>12}")
    print(f"Số câu      : {len(data):,}")
    print("=" * 55)

    # Đưa ra cảnh báo nếu dữ liệu quá mất cân bằng
    skill_pct = 100.0 * counter.get("SKILL", 0) / max(total_tokens, 1)
    if skill_pct < 5:
        print(f"⚠️ CẢNH BÁO: SKILL chỉ chiếm {skill_pct:.2f}% — RẤT mất cân bằng!")
        print("   -> Bắt buộc phải dùng class_weight hoặc focal loss ở Phase 3.")
    elif skill_pct < 15:
        print(f"⚠️ LƯU Ý: SKILL chiếm {skill_pct:.2f}% — Mất cân bằng vừa phải.")
        print("   -> Nên dùng class_weight ở Phase 3.")
    else:
        print(f"✅ TỐT: SKILL chiếm {skill_pct:.2f}% — Phân bố tương đối ổn.")

# Thực thi
label_statistics(data)

NHÃN              SỐ LƯỢNG       TỶ LỆ
-------------------------------------------------------
O                  239,984      76.54%
SKILL               73,539      23.46%
-------------------------------------------------------
TỔNG               313,523     100.00%
Số câu      : 19,323
✅ TỐT: SKILL chiếm 23.46% — Phân bố tương đối ổn.


In [11]:
# ============================================================
# PHASE 1 - CELL 6: Chia dữ liệu Train / Val / Test
# Mục đích:
# - Chia ngẫu nhiên 19,323 câu thành 3 tập: 80% Train, 10% Val, 10% Test.
# - Sử dụng Seed cố định (42) để đảm bảo kết quả có thể tái lập (reproducible).
# - Lưu 3 tập này ra file JSON để Phase 2 load lại cực nhanh (không cần parse .txt).
# ============================================================

import random
import json
from typing import Dict, List, Tuple

def split_dataset(
    data: List[Tuple[List[str], List[str]]],
    train_ratio: float = 0.8,
    val_ratio: float = 0.1,
    test_ratio: float = 0.1,
    seed: int = 42,
) -> Dict[str, List[Tuple[List[str], List[str]]]]:
    """Hàm chia dữ liệu với seed cố định."""
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-3, "Tổng tỷ lệ phải = 1.0"

    # Copy dữ liệu và xáo trộn
    data_shuffled = data.copy()
    random.Random(seed).shuffle(data_shuffled)

    n = len(data_shuffled)
    n_train = int(n * train_ratio)
    n_val   = int(n * val_ratio)
    
    # Chia tập
    train = data_shuffled[:n_train]
    val   = data_shuffled[n_train : n_train + n_val]
    test  = data_shuffled[n_train + n_val :] # Test lấy phần còn lại để không mất câu nào

    print("KẾT QUẢ CHIA TẬP:")
    print(f"  Train : {len(train):>6,} câu ({len(train)/n:.2%})")
    print(f"  Val   : {len(val):>6,} câu ({len(val)/n:.2%})")
    print(f"  Test  : {len(test):>6,} câu ({len(test)/n:.2%})")
    print(f"  TOTAL : {len(train)+len(val)+len(test):>6,} câu")
    
    return {"train": train, "validation": val, "test": test}


def save_splits_to_json(splits: dict, out_dir: Path):
    """Lưu các tập dữ liệu ra file JSON."""
    for split_name, sents in splits.items():
        # Chuyển tuple thành dict để dễ serialize JSON
        payload = [{"tokens": t, "tags": g} for t, g in sents]
        out_path = out_dir / f"{split_name}.json"
        
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(payload, f, ensure_ascii=False)
            
        print(f"[Save] Đã lưu {split_name:<10s} → {out_path.name}")


# ---- Thực thi ----
# 1. Chia tập
splits = split_dataset(
    data,
    train_ratio=Config.TRAIN_RATIO,
    val_ratio=Config.VAL_RATIO,
    test_ratio=Config.TEST_RATIO,
    seed=Config.SEED
)

# 2. Lưu ra JSON
print("\nĐang lưu file JSON...")
save_splits_to_json(splits, Config.OUTPUT_DIR)
print("Hoàn tất Phase 1!")

KẾT QUẢ CHIA TẬP:
  Train : 15,458 câu (80.00%)
  Val   :  1,932 câu (10.00%)
  Test  :  1,933 câu (10.00%)
  TOTAL : 19,323 câu

Đang lưu file JSON...
[Save] Đã lưu train      → train.json
[Save] Đã lưu validation → validation.json
[Save] Đã lưu test       → test.json
Hoàn tất Phase 1!


In [12]:
# ============================================================
# PHASE 2 - CELL 1: Khôi phục Config & Load dữ liệu
# Mục đích:
# - Đảm bảo dữ liệu Train/Val/Test được load lại chính xác từ file JSON.
# - Tránh việc phải parse lại file .txt gốc (giúp tiết kiệm thời gian).
# ============================================================

import json
from pathlib import Path
from typing import List, Tuple, Dict

# 1. Khôi phục lại class Config (để chắc chắn các biến hằng số luôn tồn tại)
class Config:
    DATA_PATH  = Path("/kaggle/input/datasets/huyende/vn-resumes-ner-txt/Vn_resumes_NER.txt")
    OUTPUT_DIR = Path("/kaggle/working")
    
    LABELS     = ["O", "SKILL"]
    LABEL2ID   = {"O": 0, "SKILL": 1}
    ID2LABEL   = {0: "O", 1: "SKILL"}
    
    MAX_LEN    = 256
    MODEL_NAME = "vinai/phobert-base"

# 2. Load dữ liệu từ 3 file JSON (đã lưu ở Phase 1)
splits = {}
try:
    for split_name in ["train", "validation", "test"]:
        json_path = Config.OUTPUT_DIR / f"{split_name}.json"
        
        with open(json_path, "r", encoding="utf-8") as f:
            raw_data = json.load(f)
            
        # Chuyển đổi từ dạng dict {"tokens": [...], "tags": [...]} về dạng Tuple ban đầu
        splits[split_name] = [(item["tokens"], item["tags"]) for item in raw_data]
        
    print("[✓] Đã load dữ liệu thành công từ file JSON:")
    for k, v in splits.items():
        print(f"    - Tập {k:10s}: {len(v):,} câu")
        
except FileNotFoundError as e:
    print(f"[X] LỖI: Không tìm thấy file JSON. Bạn đã chạy xong Phase 1 chưa? Chi tiết: {e}")

[✓] Đã load dữ liệu thành công từ file JSON:
    - Tập train     : 15,458 câu
    - Tập validation: 1,932 câu
    - Tập test      : 1,933 câu


In [2]:
# ============================================================
# CELL 7 (FIXED): Load PhoBERT tokenizer (chấp nhận bản Slow)
# - Chúng ta sẽ tự xử lý word_ids thủ công ở Cell 8.
# ============================================================

import transformers
from transformers import AutoTokenizer

print(f"[Env] transformers version: {transformers.__version__}")
MODEL_NAME = "vinai/phobert-base"

# Load tokenizer (sẽ trả về PhobertTokenizer - Slow)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

print(f"[✓] Tokenizer class : {type(tokenizer).__name__}")
print(f"[✓] Vocab size      : {tokenizer.vocab_size:,}")
print(f"[✓] model_max_length: {tokenizer.model_max_length}")
print(f"[✓] Special tokens  : {tokenizer.special_tokens_map}")

[Env] transformers version: 5.22.2
[✓] Tokenizer class : PhobertTokenizer
[✓] Vocab size      : 64,000
[✓] model_max_length: 1000000000000000019884624838656
[✓] Special tokens  : {'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}


In [13]:
# ============================================================
# PHASE 2 - CELL 2: Khởi tạo PhoBERT Tokenizer
# Mục đích:
# - Load tokenizer "vinai/phobert-base".
# - Do thư viện transformers hiện tại chưa hỗ trợ Fast Tokenizer cho PhoBERT,
#   ta bắt buộc phải dùng bản Slow (use_fast=False).
# - Sẽ tự xử lý word_ids bằng code thủ công ở Cell tiếp theo.
# ============================================================

from transformers import AutoTokenizer
import transformers

print(f"[Env] Phiên bản transformers: {transformers.__version__}")

# Khởi tạo tokenizer (bản Slow)
tokenizer = AutoTokenizer.from_pretrained(
    Config.MODEL_NAME, 
    use_fast=False
)

print(f"[✓] Tokenizer class : {type(tokenizer).__name__}")
print(f"[✓] Vocab size      : {tokenizer.vocab_size:,}")
print(f"[✓] Max length      : {tokenizer.model_max_length}")

# Demo nhanh khả năng tách từ của PhoBERT
demo_words = ["lập_trình", "sinh_viên", "python", "c", "#"]
print(f"\nDemo tách từ với list: {demo_words}")
print(f"Kết quả: {tokenizer.tokenize(' '.join(demo_words))}")

[Env] Phiên bản transformers: 5.22.2
[✓] Tokenizer class : PhobertTokenizer
[✓] Vocab size      : 64,000
[✓] Max length      : 1000000000000000019884624838656

Demo tách từ với list: ['lập_trình', 'sinh_viên', 'python', 'c', '#']
Kết quả: ['lập_trình', 'sinh_viên', 'py@@', 'thon', 'c', '#']


In [15]:
# ============================================================
# PHASE 2 - CELL 3 (FIXED): Căn chỉnh nhãn (Label Alignment)
# - Sửa lỗi KeyError bằng cách ép kiểu nhãn (tag) về integer ID.
# ============================================================

from typing import List, Tuple, Dict

LABEL_PAD_TOKEN_ID = -100

def get_manual_word_ids(tokenizer, words: List[str]) -> List[int]:
    """Tạo word_ids thủ công cho Slow Tokenizer."""
    word_ids = [None]  # Token <s> (CLS)
    for idx, word in enumerate(words):
        sub_tokens = tokenizer.tokenize(word)
        word_ids.extend([idx] * len(sub_tokens))
    word_ids.append(None)  # Token </s> (SEP)
    return word_ids


def tokenize_and_align_batch(
    sentences: List[Tuple[List[str], List[str]]],
    tokenizer,
    max_len: int,
    label_pad_token_id: int = LABEL_PAD_TOKEN_ID,
) -> Dict[str, List]:
    """Tokenize batch câu + align nhãn."""
    words_batch = [toks for toks, _ in sentences]
    tags_batch  = [tags for _, tags in sentences]

    # 1. Tokenize toàn bộ câu
    encodings = tokenizer(
        words_batch,
        is_split_into_words=True,
        truncation=True,
        max_length=max_len,
        padding=False,
        return_attention_mask=True,
    )

    all_labels = []
    
    # 2. Duyệt từng câu để align nhãn
    for i, tags in enumerate(tags_batch):
        word_ids = get_manual_word_ids(tokenizer, words_batch[i])
        
        if len(word_ids) > len(encodings["input_ids"][i]):
            word_ids = word_ids[:len(encodings["input_ids"][i])]

        aligned_labels = []
        prev_wid = -1
        
        for wid in word_ids:
            if wid is None:
                aligned_labels.append(label_pad_token_id)
            elif wid == prev_wid:
                aligned_labels.append(label_pad_token_id)
            else:
                # FIX LỖI: Ép kiểu nhãn về ID số nguyên (0 hoặc 1)
                raw_tag = tags[wid]
                if isinstance(raw_tag, str):
                    # Nếu file JSON lưu dạng chuỗi 'O', 'SKILL'
                    tag_id = Config.LABEL2ID[raw_tag]
                else:
                    # Nếu file JSON lưu dạng số '0', '1' hoặc 0, 1
                    tag_id = int(raw_tag)
                
                aligned_labels.append(tag_id)
                
            prev_wid = wid if wid is not None else -1
            
        all_labels.append(aligned_labels)

    encodings["labels"] = all_labels
    return encodings


# ============================================================
# TEST LẠI ALIGNMENT (ĐÃ SỬA LỖI)
# ============================================================
print("=" * 60)
print("TEST CĂN CHỈNH NHÃN (2 CÂU ĐẦU TIÊN)")
print("=" * 60)

test_batch = tokenize_and_align_batch(
    sentences=splits["train"][:2], 
    tokenizer=tokenizer, 
    max_len=Config.MAX_LEN
)

for i in range(len(test_batch["input_ids"])):
    toks = tokenizer.convert_ids_to_tokens(test_batch["input_ids"][i])
    lbls = test_batch["labels"][i]
    
    print(f"\n--- Câu #{i} ---")
    for j, (t, l) in enumerate(zip(toks, lbls)):
        if j > 25: 
            print("... (cắt bớt để hiển thị)")
            break
            
        # Dùng .get() để không bao giờ bị KeyError nếu có nhãn lạ
        l_name = "IGNORE" if l == LABEL_PAD_TOKEN_ID else Config.ID2LABEL.get(l, str(l))
        flag = " ★" if l != LABEL_PAD_TOKEN_ID else ""
        print(f"{j:<5}{t:<18}{l_name}{flag}")

TEST CĂN CHỈNH NHÃN (2 CÂU ĐẦU TIÊN)

--- Câu #0 ---
0    <s>               IGNORE
1    de@@              O ★
2    ve@@              IGNORE
3    lop@@             IGNORE
4    ment              IGNORE
5    en@@              O ★
6    vir@@             IGNORE
7    on@@              IGNORE
8    ment              IGNORE
9    de@@              O ★
10   ve@@              IGNORE
11   lop@@             IGNORE
12   ment              IGNORE
13   to@@              O ★
14   ol@@              IGNORE
15   s                 IGNORE
16   ec@@              SKILL ★
17   lip@@             IGNORE
18   se                IGNORE
19   al@@              SKILL ★
20   to@@              IGNORE
21   va_@@             IGNORE
22   x@@               IGNORE
23   m@@               IGNORE
24   l@@               IGNORE
25   sp@@              IGNORE
... (cắt bớt để hiển thị)

--- Câu #1 ---
0    <s>               IGNORE
1    biết              O ★
2    xử_lý             SKILL ★
3    các               SKILL ★
4    sự_cố      

In [16]:
# ============================================================
# PHASE 2 - CELL 4: PyTorch Dataset, DataCollator & DataLoader
# Mục đích:
# - Tạo class ResumeNERDataset để PyTorch có thể lặp (iterate) qua dữ liệu.
# - Tối ưu: Tokenize & align nhãn 1 LẦN DUY NHẤT trong __init__.
# - Sử dụng DataCollator để tự động pad (đệm) các câu ngắn trong batch 
#   cho bằng câu dài nhất (Dynamic Padding) giúp tiết kiệm VRAM.
# ============================================================

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DataCollatorForTokenClassification

class ResumeNERDataset(Dataset):
    """Dataset PyTorch cho bài toán NER với PhoBERT."""
    def __init__(self, sentences, tokenizer, max_len):
        # Gọi hàm align nhãn đã viết ở Cell 3
        self.encodings = tokenize_and_align_batch(
            sentences=sentences, 
            tokenizer=tokenizer, 
            max_len=max_len
        )
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        # Trả về dictionary chứa tensor của 1 câu
        return {
            "input_ids": torch.tensor(self.encodings["input_ids"][idx], dtype=torch.long),
            "attention_mask": torch.tensor(self.encodings["attention_mask"][idx], dtype=torch.long),
            "labels": torch.tensor(self.encodings["labels"][idx], dtype=torch.long),
        }

# ---- 1. Khởi tạo Dataset ----
print("Đang khởi tạo Dataset (quá trình này sẽ mất vài giây)...")
train_dataset = ResumeNERDataset(splits["train"], tokenizer, Config.MAX_LEN)
val_dataset   = ResumeNERDataset(splits["validation"], tokenizer, Config.MAX_LEN)
test_dataset  = ResumeNERDataset(splits["test"], tokenizer, Config.MAX_LEN)

print(f"[✓] Train Dataset: {len(train_dataset):,} câu")
print(f"[✓] Val Dataset  : {len(val_dataset):,} câu")
print(f"[✓] Test Dataset : {len(test_dataset):,} câu")

# ---- 2. Khởi tạo DataCollator ----
# Tự động pad các tensor trong batch cho bằng nhau
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding="longest",               # Pad theo câu dài nhất trong batch
    label_pad_token_id=LABEL_PAD_TOKEN_ID, # Pad nhãn bằng -100
    return_tensors="pt"
)

# ---- 3. Khởi tạo DataLoader ----
BATCH_SIZE = 16

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=data_collator)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator)

print(f"\n[✓] Đã tạo DataLoader với Batch Size = {BATCH_SIZE}:")
print(f"    Train : {len(train_loader):,} batches")
print(f"    Val   : {len(val_loader):,} batches")
print(f"    Test  : {len(test_loader):,} batches")

# ---- 4. Sanity Check (Kiểm tra 1 batch thực tế) ----
print("\n--- Kiểm tra 1 Batch thực tế ---")
batch = next(iter(train_loader))
for k, v in batch.items():
    print(f"  {k:<15}: shape={tuple(v.shape)} | dtype={v.dtype}")

Đang khởi tạo Dataset (quá trình này sẽ mất vài giây)...
[✓] Train Dataset: 15,458 câu
[✓] Val Dataset  : 1,932 câu
[✓] Test Dataset : 1,933 câu

[✓] Đã tạo DataLoader với Batch Size = 16:
    Train : 967 batches
    Val   : 121 batches
    Test  : 121 batches

--- Kiểm tra 1 Batch thực tế ---
  input_ids      : shape=(16, 36) | dtype=torch.int64
  attention_mask : shape=(16, 36) | dtype=torch.int64
  labels         : shape=(16, 36) | dtype=torch.int64


In [17]:
# ============================================================
# PHASE 2 - CELL 5: Tính Class Weights & Lưu Meta cho Phase 3
# Mục đích:
# - Tính trọng số (weight) cho từng nhãn dựa trên tần suất xuất hiện.
#   Công thức: w_i = Tổng số token / (Số nhãn * Số lượng token của nhãn i)
# - Các nhãn hiếm (SKILL) sẽ có weight cao hơn, giúp CrossEntropyLoss
#   phạt nặng hơn nếu model đoán sai nhãn này.
# - Lưu các thông số cần thiết ra file để Phase 3 load lại.
# ============================================================

from collections import Counter

def compute_class_weights(encodings, num_labels=2):
    """Tính class weights từ tập Train."""
    counter = Counter()
    
    # Duyệt qua toàn bộ nhãn, bỏ qua -100 (các token bị ignore)
    for lbls in encodings["labels"]:
        for l in lbls:
            if l != LABEL_PAD_TOKEN_ID:
                counter[l] += 1
                
    total_tokens = sum(counter.values())
    
    # w_i = total / (num_labels * count_i)
    weights = [
        total_tokens / (num_labels * max(counter.get(i, 1), 1)) 
        for i in range(num_labels)
    ]
    
    return torch.tensor(weights, dtype=torch.float), counter

# ---- 1. Tính toán weights từ tập Train ----
weights, counter = compute_class_weights(train_dataset.encodings, num_labels=2)

print("=" * 45)
print("CLASS WEIGHTS (TẬP TRAIN)")
print("=" * 45)
print(f"{'Nhãn':<10}{'Số lượng':>14}{'Weight':>12}")
print("-" * 45)

for i, lbl in Config.ID2LABEL.items():
    print(f"{lbl:<10}{counter[i]:>14,}{weights[i]:>12.4f}")
    
print("-" * 45)
print(f"👉 Weight O = {weights[0]:.4f}, Weight SKILL = {weights[1]:.4f}")

# ---- 2. Lưu Meta Data ----
meta_path = Config.OUTPUT_DIR / "phase2_meta.pt"
torch.save({
    "model_name": Config.MODEL_NAME,
    "max_len": Config.MAX_LEN,
    "label2id": Config.LABEL2ID,
    "id2label": Config.ID2LABEL,
    "class_weights": weights,
    "label_counts": dict(counter),
}, meta_path)

print(f"\n[✓] Đã lưu file meta → {meta_path.name}")
print(f"    Dung lượng: {meta_path.stat().st_size / 1024:.1f} KB")
print("Chuẩn bị sẵn sàng cho Phase 3: Huấn luyện mô hình!")

CLASS WEIGHTS (TẬP TRAIN)
Nhãn            Số lượng      Weight
---------------------------------------------
O                191,204      0.6536
SKILL             58,738      2.1276
---------------------------------------------
👉 Weight O = 0.6536, Weight SKILL = 2.1276

[✓] Đã lưu file meta → phase2_meta.pt
    Dung lượng: 1.8 KB
Chuẩn bị sẵn sàng cho Phase 3: Huấn luyện mô hình!


In [20]:
# ============================================================
# PHASE 3 - CELL 1 (FIXED): Khởi tạo Model, Custom Trainer & Args
# - Đã sửa evaluation_strategy -> eval_strategy theo chuẩn transformers 4.46+
# ============================================================

import torch
from pathlib import Path
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer

# 1. Khôi phục lại class Config đầy đủ thuộc tính
class Config:
    OUTPUT_DIR = Path("/kaggle/working")
    LABELS     = ["O", "SKILL"]
    LABEL2ID   = {"O": 0, "SKILL": 1}
    ID2LABEL   = {0: "O", 1: "SKILL"}
    SEED       = 42
    MODEL_NAME = "vinai/phobert-base"

# 2. Load Meta Data (Class weights) từ Phase 2
meta_path = Config.OUTPUT_DIR / "phase2_meta.pt"
meta = torch.load(meta_path, map_location="cpu")
class_weights = meta["class_weights"].to("cuda" if torch.cuda.is_available() else "cpu")
print(f"[✓] Đã load class weights: {class_weights.tolist()}")

# 3. Khởi tạo mô hình PhoBERT
model = AutoModelForTokenClassification.from_pretrained(
    Config.MODEL_NAME,
    num_labels=len(Config.LABELS),
    id2label=Config.ID2LABEL,
    label2id=Config.LABEL2ID,
)

# 4. Tạo CustomTrainer để truyền class_weights vào hàm Loss
class CustomTrainer(Trainer):
    def __init__(self, class_weights=None, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        loss_fct = torch.nn.CrossEntropyLoss(
            weight=self.class_weights, 
            ignore_index=-100
        )
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        
        return (loss, outputs) if return_outputs else loss

# 5. Cấu hình tham số huấn luyện (ĐÃ SỬA eval_strategy)
training_args = TrainingArguments(
    output_dir=str(Config.OUTPUT_DIR / "checkpoints"),
    learning_rate=2e-5,                     
    per_device_train_batch_size=16,         
    per_device_eval_batch_size=16,
    num_train_epochs=3,                     
    weight_decay=0.01,                      
    eval_strategy="epoch",                  # ✅ Đã đổi tên chuẩn cho transformers 4.46+
    save_strategy="epoch",                  
    logging_steps=100,                      
    load_best_model_at_end=True,            
    metric_for_best_model="f1",             
    fp16=True,                              
    report_to="none",                       
    seed=Config.SEED                        
)

print("[✓] Đã khởi tạo mô hình PhoBERT và cấu hình TrainingArguments thành công.")

[✓] Đã load class weights: [0.6536003351211548, 2.1276004314422607]


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForTokenClassification LOAD REPORT from: vinai/phobert-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.decoder.bias        | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[✓] Đã khởi tạo mô hình PhoBERT và cấu hình TrainingArguments thành công.


In [21]:
# ============================================================
# PHASE 3 - CELL 2: Hàm Đánh giá (Metrics) & Bắt đầu Train
# Mục đích:
# - Định nghĩa hàm compute_metrics để tính F1-score, Precision, Recall 
#   (sử dụng thư viện seqeval đã cài ở Phase 1).
# - Khởi tạo CustomTrainer với đầy đủ tham số.
# - Gọi lệnh trainer.train() để bắt đầu quá trình huấn luyện.
# ============================================================

import numpy as np
from seqeval.metrics import precision_score, recall_score, f1_score, accuracy_score, classification_report

def compute_metrics(eval_pred):
    """Hàm tính toán các chỉ số đánh giá trong quá trình Eval."""
    predictions, labels = eval_pred
    # Lấy nhãn có xác suất cao nhất (argmax)
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    # Duyệt qua từng batch trong kết quả dự đoán
    for pred_seq, label_seq in zip(predictions, labels):
        cur_preds = []
        cur_labels = []
        for p, l in zip(pred_seq, label_seq):
            # Bỏ qua các token padding và special tokens có nhãn -100
            if l != -100:
                cur_preds.append(Config.ID2LABEL[p])
                cur_labels.append(Config.ID2LABEL[l])
        
        true_predictions.append(cur_preds)
        true_labels.append(cur_labels)

    # Tính toán các metrics
    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
        "accuracy": accuracy_score(true_labels, true_predictions),
    }

# ---- 1. Khởi tạo Trainer ----
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights  # Truyền class_weights vào CustomTrainer
)

# ---- 2. Bắt đầu Huấn luyện ----
print("=" * 60)
print("BẮT ĐẦU QUÁ TRÌNH HUẤN LUYỆN...")
print("=" * 60)

# Lệnh này sẽ chạy 3 Epochs. Trên GPU T4, thời gian dự kiến khoảng 15-25 phút.
train_result = trainer.train()

# ---- 3. Đánh giá lần cuối trên tập Validation ----
print("\n" + "=" * 60)
print("ĐÁNH GIÁ MÔ HÌNH TỐT NHẤT TRÊN TẬP VALIDATION")
print("=" * 60)
eval_results = trainer.evaluate()
print(eval_results)

BẮT ĐẦU QUÁ TRÌNH HUẤN LUYỆN...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.140227,0.121595,0.847606,0.966906,0.903334,0.949540
2,0.111186,0.116731,0.881568,0.956672,0.917586,0.958096
3,0.099314,0.113560,0.882677,0.956938,0.918309,0.958485


/usr/local/lib/python3.13/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: SKILL seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: SKILL seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (2 outputs left out)


ĐÁNH GIÁ MÔ HÌNH TỐT NHẤT TRÊN TẬP VALIDATION


/usr/local/lib/python3.13/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: SKILL seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.099314,0.113560,3,0.882677,0.956938,0.918309,0.958485


{'eval_loss': 0.11355990171432495, 'eval_precision': 0.8826774549466716, 'eval_recall': 0.9569377990430622, 'eval_f1': 0.9183087813277214, 'eval_accuracy': 0.9584845735027223}


In [22]:
# ============================================================
# PHASE 3 - CELL 3: Đánh giá trên tập Test & Lưu mô hình
# Mục đích:
# - Chạy dự đoán (predict) trên tập Test (1,933 câu) để có kết quả 
#   khách quan nhất (vì tập Test chưa từng được dùng để train/eval).
# - In báo cáo chi tiết (Classification Report) để xem Precision/Recall của từng nhãn.
# - Lưu mô hình và tokenizer xuống ổ cứng để tái sử dụng (inference) sau này.
# ============================================================

import numpy as np
from seqeval.metrics import classification_report

print("Đang đánh giá mô hình trên tập Test...")
# 1. Dự đoán trên tập Test
test_results = trainer.predict(test_dataset)

# In các chỉ số tổng quan
print("\n[KẾT QUẢ TỔNG QUAN TRÊN TẬP TEST]")
for k, v in test_results.metrics.items():
    if "runtime" not in k and "samples" not in k:
        print(f"  {k:<15}: {v:.4f}")

# 2. Lấy nhãn dự đoán và nhãn thực tế để in báo cáo chi tiết
predictions = np.argmax(test_results.predictions, axis=2)
labels = test_results.label_ids

true_predictions = []
true_labels = []

for pred_seq, label_seq in zip(predictions, labels):
    cur_preds = []
    cur_labels = []
    for p, l in zip(pred_seq, label_seq):
        # Bỏ qua padding tokens (-100)
        if l != -100:
            cur_preds.append(Config.ID2LABEL[p])
            cur_labels.append(Config.ID2LABEL[l])
    true_predictions.append(cur_preds)
    true_labels.append(cur_labels)

# 3. In Classification Report chi tiết
print("\n" + "=" * 60)
print("CHI TIẾT KẾT QUẢ TRÊN TẬP TEST (Classification Report)")
print("=" * 60)
# Giả lập cảnh báo của seqeval bằng cách tắt warnings trong block này
import warnings
warnings.filterwarnings("ignore")
print(classification_report(true_labels, true_predictions, digits=4))

# 4. Lưu mô hình tốt nhất
save_path = Config.OUTPUT_DIR / "best_phobert_ner"
trainer.save_model(str(save_path))
tokenizer.save_pretrained(str(save_path))

print(f"\n[✓] Đã lưu mô hình và tokenizer thành công tại:\n    {save_path}")

Đang đánh giá mô hình trên tập Test...



[KẾT QUẢ TỔNG QUAN TRÊN TẬP TEST]
  test_loss      : 0.1093
  test_precision : 0.8778
  test_recall    : 0.9635
  test_f1        : 0.9187
  test_accuracy  : 0.9619
  test_steps_per_second: 8.2360

CHI TIẾT KẾT QUẢ TRÊN TẬP TEST (Classification Report)
              precision    recall  f1-score   support

        KILL     0.8778    0.9635    0.9187      7256

   micro avg     0.8778    0.9635    0.9187      7256
   macro avg     0.8778    0.9635    0.9187      7256
weighted avg     0.8778    0.9635    0.9187      7256



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


[✓] Đã lưu mô hình và tokenizer thành công tại:
    /kaggle/working/best_phobert_ner
